# Zero-shot retrieval — PE-Core-G14-448 only

Pipeline đơn giản nhất, **không train gì cả**: dùng Meta `PE-Core-G14-448` (1.8B params) zero-shot để encode 36,773 gallery ảnh và 1,978 query captions, cosine sim → top-10 → `answer.txt`.

**Bước:**
1. Setup Drive + dataset (qua `aic_colab_utils.setup_aic2026_environment`).
2. Clone `facebookresearch/perception_models` repo, load PE-G14-448 (BF16 + channels_last).
3. Encode gallery ảnh (batched, NPZ cache để resume).
4. Encode queries theo **đúng order của `query_index.txt`** (BTC chấm theo order này, không phải order JSON).
5. Cosine similarity (1978, 36773), top-10 indices → gallery filename stems.
6. Ghi `answer.txt` → zip `answer.zip` (theo format mẫu `Document/answer.txt`).

**Format submission** (xác minh từ `https://github.com/Shuyu-XJTU/PAB-for-ECCV26-Workshop-Track4`):
- 1978 dòng tương ứng 1978 query trong `query_index.txt`
- Mỗi dòng = 10 gallery filename stems (15 ký tự alphanumeric) space-separated, không có extension
- Zip `answer.zip` chứa duy nhất `answer.txt`

**ETA A100 40GB BF16**: ~5-10 min (gallery encode chi phối).


In [ ]:
# === Bootstrap ===
from pathlib import Path
import os, sys, json, shutil, subprocess, time, zipfile, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

NOTEBOOK_DIR = Path('.').resolve()
UTILS_DIR = NOTEBOOK_DIR.parent / 'utils'  # aic26/utils/
for _p in (str(UTILS_DIR), str(NOTEBOOK_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    save_npz_atomic, l2_normalize_np,
    sync_chunk_to_drive, wait_for_pending_syncs, maybe_clear_cuda,
)

PATHS = setup_aic2026_environment()
# Đọc trực tiếp từ Drive FUSE — KHÔNG copy local. DataLoader sẽ parallel
# (16 workers + prefetch=4) để amortize per-file latency của Drive FUSE.
# Trade-off: GPU idle ~40-60% chờ I/O nhưng không tốn upfront copy time.

INPUT_ROOT       = PATHS['input_root']
LOCAL_ROOT       = PATHS['local_root']
DRIVE_ROOT       = PATHS['drive_root']
DRIVE_OUTPUT_ROOT= PATHS['drive_output_root']
TEST_DIR         = PATHS['test_dir']        # /content/aic_local/raw/name-masked_test-set
GALLERY_DIR      = PATHS['gallery_dir']     # .../gallery/gallery (nested) hoặc /gallery
print('TEST_DIR    :', TEST_DIR)
print('GALLERY_DIR :', GALLERY_DIR)

device = select_a100_device()

In [ ]:
# === Locate query files ===
# Dataset có `query_text.json` (JSONL, fields: query_index, caption, change)
# và `query_index.txt` (1978 line: query_index thuần, định nghĩa ORDER OF SUBMISSION).
QUERY_JSON = TEST_DIR / 'query_text.json'
QUERY_INDEX_TXT = TEST_DIR / 'query_index.txt'
assert QUERY_JSON.exists(),       f'missing {QUERY_JSON}'
assert QUERY_INDEX_TXT.exists(),  f'missing {QUERY_INDEX_TXT}'

# Build {query_index → caption}
qid_to_caption = {}
with open(QUERY_JSON, encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        qid_to_caption[obj['query_index']] = obj['caption']
print(f'JSON queries: {len(qid_to_caption)}')

# Submission order = order of query_index.txt
submission_qids = [ln.strip() for ln in QUERY_INDEX_TXT.read_text().splitlines() if ln.strip()]
assert len(submission_qids) == 1978, f'expected 1978 queries, got {len(submission_qids)}'
missing = [q for q in submission_qids if q not in qid_to_caption]
assert not missing, f'{len(missing)} queries in index but not in JSON (first: {missing[:3]})'
submission_captions = [qid_to_caption[q] for q in submission_qids]
print(f'submission ordering: {len(submission_qids)} queries (matches query_index.txt order)')


In [ ]:
# === Pinned gallery ordering (shell ls — tránh iterdir Errno 5 trên Drive FUSE) ===
EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

def _ls_dir(path, retries=5, sleep=10):
    """Liệt kê file trong path qua shell `ls`. Drive FUSE friendly hơn nhiều
    so với Python iterdir() (hay raise Errno 5 trên folder nhiều file).
    """
    last = None
    for attempt in range(retries):
        try:
            out = subprocess.check_output(
                ['bash', '-c', f'ls -1 "{path}" 2>/dev/null'],
                timeout=600, text=True,
            )
            names = [n.strip() for n in out.splitlines() if n.strip()]
            if names:
                return names
        except Exception as exc:
            last = exc
            print(f'  ls attempt {attempt+1}/{retries} failed: {exc}; retry in {sleep}s')
            time.sleep(sleep)
    raise RuntimeError(f'shell ls failed after {retries} attempts: {last}')

print(f'enumerating {GALLERY_DIR} (this may take 1-3 min on Drive FUSE)...')
t0 = time.time()
names = _ls_dir(GALLERY_DIR)
print(f'  → {len(names)} entries in {time.time()-t0:.0f}s')

gallery_paths = sorted(GALLERY_DIR / n for n in names if Path(n).suffix.lower() in EXTS)
gallery_ids = [p.stem for p in gallery_paths]
print(f'gallery N = {len(gallery_paths)} (expect 36,773)')
print('sample IDs:', gallery_ids[:3])


In [ ]:
# === Install Meta perception_models repo (PE-Core-G14-448) ===
PM_REPO = LOCAL_ROOT / 'perception_models'
if not PM_REPO.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/facebookresearch/perception_models.git',
                           str(PM_REPO)])
if str(PM_REPO) not in sys.path:
    sys.path.insert(0, str(PM_REPO))

# Minimal deps (perception_models has its own requirements.txt; install lightly)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'timm', 'ftfy', 'regex', 'tokenizers', 'einops'])
print('perception_models ready at', PM_REPO)


In [ ]:
# === Load PE-Core-G14-448 ===
import torch
import torch.nn.functional as F
from PIL import Image
from core.vision_encoder import pe
from core.vision_encoder import transforms as pe_transforms

PE_MODEL_NAME = 'PE-Core-G14-448'

maybe_clear_cuda()
print('loading', PE_MODEL_NAME)
pe_model = pe.CLIP.from_config(PE_MODEL_NAME, pretrained=True).to(device).eval()
pe_model = pe_model.to(memory_format=torch.channels_last)

pe_preprocess = pe_transforms.get_image_transform(pe_model.image_size)
pe_tokenizer  = pe_transforms.get_text_tokenizer(pe_model.context_length)
_fallback_size = pe_model.image_size if isinstance(pe_model.image_size, int) else pe_model.image_size[0]
pe_fallback = pe_preprocess(Image.new('RGB', (_fallback_size, _fallback_size))).clone()
print(f'image_size={pe_model.image_size} | context_length={pe_model.context_length}')


In [ ]:
# === Batched gallery encode (NPZ cache for resume) ===
from torch.utils.data import Dataset, DataLoader
import numpy as np

# VRAM-driven batch: A100 40GB ≈ 96, 80GB ≈ 256.
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
IMG_BATCH = 96 if total_gb < 60 else 256
# Drive FUSE I/O bottleneck — tăng workers + prefetch để parallel-fetch
# nhiều ảnh cùng lúc, amortize per-file latency.
NUM_WORKERS = 16
PREFETCH_FACTOR = 4

OUT_LOCAL = LOCAL_ROOT / 'zero_shot_pe_g14'
OUT_DRIVE = DRIVE_OUTPUT_ROOT / 'zero_shot_pe_g14'
OUT_LOCAL.mkdir(parents=True, exist_ok=True)
OUT_DRIVE.mkdir(parents=True, exist_ok=True)

GALLERY_NPZ = OUT_LOCAL / 'gallery_emb.npz'

class GalleryDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        # Retry once on transient Drive FUSE Errno 5
        for attempt in range(2):
            try:
                tensor = pe_preprocess(Image.open(self.paths[i]).convert('RGB'))
                return tensor, i, True
            except Exception:
                if attempt == 0:
                    time.sleep(0.5)
                    continue
        return pe_fallback.clone(), i, False

@torch.inference_mode()
def encode_gallery():
    # Resume from Drive cache if present
    drive_cache = OUT_DRIVE / 'gallery_emb.npz'
    if not GALLERY_NPZ.exists() and drive_cache.exists():
        shutil.copy2(drive_cache, GALLERY_NPZ)
    if GALLERY_NPZ.exists():
        cached = np.load(GALLERY_NPZ, allow_pickle=False)
        if list(cached['ids']) == gallery_ids and cached['emb'].shape[0] == len(gallery_ids):
            print(f'reuse cached gallery emb: {cached["emb"].shape}')
            return cached['emb']
        else:
            print('cache mismatch — re-encoding')

    ds = GalleryDataset(gallery_paths)
    dl = DataLoader(
        ds,
        batch_size=IMG_BATCH,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=PREFETCH_FACTOR,
        persistent_workers=True,
    )
    feats = np.empty((len(gallery_paths), 1280), dtype=np.float16)
    ok_arr = np.zeros(len(gallery_paths), dtype=bool)
    t0 = time.time()
    seen = 0
    for tensors, idxs, oks in dl:
        tensors = tensors.to(device, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            out = pe_model.encode_image(tensors)
        out = F.normalize(out.float(), dim=-1).half().cpu().numpy()
        for j, gid in enumerate(idxs.tolist()):
            feats[gid] = out[j]
            ok_arr[gid] = bool(oks[j])
        seen += tensors.size(0)
        if seen % (IMG_BATCH * 10) == 0:
            elapsed = time.time() - t0
            rate = seen / elapsed
            eta = (len(gallery_paths) - seen) / rate if rate > 0 else 0
            print(f'  encoded {seen}/{len(gallery_paths)} | {rate:.0f} img/s | '
                  f'elapsed {elapsed:.0f}s | ETA {eta:.0f}s')
    print(f'gallery encode done in {time.time()-t0:.0f}s | n_fail={int((~ok_arr).sum())}')

    save_npz_atomic(GALLERY_NPZ, ids=np.array(gallery_ids), emb=feats, ok=ok_arr)
    sync_chunk_to_drive(GALLERY_NPZ, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
    return feats

gallery_emb_np = encode_gallery()    # (G, 1280) fp16, L2-normalised
print('gallery_emb_np:', gallery_emb_np.shape, gallery_emb_np.dtype)


In [ ]:
# === Query text encode (order = query_index.txt order) ===
TEXT_BATCH = 256

@torch.inference_mode()
def encode_texts(captions):
    out_list = []
    for s in range(0, len(captions), TEXT_BATCH):
        chunk = captions[s:s+TEXT_BATCH]
        tokens = pe_tokenizer(chunk).to(device)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            feats = pe_model.encode_text(tokens)
        feats = F.normalize(feats.float(), dim=-1).half().cpu().numpy()
        out_list.append(feats)
    return np.concatenate(out_list, axis=0)

t0 = time.time()
query_emb_np = encode_texts(submission_captions)
print(f'query_emb_np: {query_emb_np.shape} in {time.time()-t0:.1f}s')


In [ ]:
# === Cosine sim + top-10 (chunked to avoid 1.4GB FP32 sim matrix) ===
G_t = torch.from_numpy(gallery_emb_np).to(device, dtype=torch.float32)   # (G, D)
Q_t = torch.from_numpy(query_emb_np).to(device, dtype=torch.float32)     # (Q, D)
# Already L2-normalised → matmul == cosine.

TOPK = 10
Q_CHUNK = 256
top10_indices = np.empty((Q_t.size(0), TOPK), dtype=np.int64)
for s in range(0, Q_t.size(0), Q_CHUNK):
    e = min(s + Q_CHUNK, Q_t.size(0))
    sims = Q_t[s:e] @ G_t.T                              # (chunk, G)
    _, idx = torch.topk(sims, k=TOPK, dim=1)
    top10_indices[s:e] = idx.cpu().numpy()
print('top10_indices:', top10_indices.shape)


In [ ]:
# === Sanity: confirm same top-10 as Milvus L2 method (AIC2025 ui-streamlit pattern) ===
# Last year's ui-streamlit.ipynb used Milvus with metric_type='L2':
#   similarity = 1 - max(0, hit.distance)
# With L2-normalised vectors:
#   L2_dist = sqrt(2 - 2·cos)  →  sim_milvus = 1 - sqrt(2 - 2·cos)
# That is a monotonic-increasing transform of cosine, so argsort should agree.
# Here we recompute via the Milvus formula and verify top-10 IDs match per row.

top10_milvus = np.empty_like(top10_indices)
for s in range(0, Q_t.size(0), Q_CHUNK):
    e = min(s + Q_CHUNK, Q_t.size(0))
    cos = Q_t[s:e] @ G_t.T                                  # (chunk, G)
    l2  = torch.sqrt((2.0 - 2.0 * cos).clamp(min=0.0))      # = ||q − g||
    sim_milvus = 1.0 - l2                                   # last year's score
    _, idx = torch.topk(sim_milvus, k=TOPK, dim=1)
    top10_milvus[s:e] = idx.cpu().numpy()

# Set-match (order-independent): must be 100% because monotonic transform
set_match = sum(set(a.tolist()) == set(b.tolist())
                for a, b in zip(top10_indices, top10_milvus))
# Order-match: should also be 100% modulo FP tie-breaking on near-equal scores
order_match = sum(int((a == b).all()) for a, b in zip(top10_indices, top10_milvus))

print(f'cosine vs Milvus-L2 top-10 SET   match : {set_match}/{len(top10_indices)}'
      f' ({100*set_match/len(top10_indices):.2f}%)')
print(f'cosine vs Milvus-L2 top-10 ORDER match : {order_match}/{len(top10_indices)}'
      f' ({100*order_match/len(top10_indices):.2f}%)')
assert set_match == len(top10_indices), (
    'Top-10 SET differs between cosine and Milvus-L2 — should be impossible with '
    'L2-normalised embeddings. Check normalisation.'
)
print('✓ confirmed: ranking identical to AIC2025 ui-streamlit Milvus L2 backend')


In [ ]:
# === Write answer.txt + answer.zip (BTC format) ===
SUB_LOCAL = LOCAL_ROOT / 'submission_zero_shot'; SUB_LOCAL.mkdir(parents=True, exist_ok=True)
SUB_DRIVE = DRIVE_OUTPUT_ROOT / 'submission_zero_shot'; SUB_DRIVE.mkdir(parents=True, exist_ok=True)

ANSWER_TXT = SUB_LOCAL / 'answer.txt'
ANSWER_ZIP = SUB_LOCAL / 'answer.zip'

with open(ANSWER_TXT, 'w', encoding='utf-8') as f:
    for row in top10_indices:
        f.write(' '.join(gallery_ids[i] for i in row) + '\n')
print(f'wrote {ANSWER_TXT}  ({ANSWER_TXT.stat().st_size} bytes)')

with zipfile.ZipFile(ANSWER_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ANSWER_TXT, arcname='answer.txt')
print(f'wrote {ANSWER_ZIP}')

shutil.copy2(ANSWER_TXT, SUB_DRIVE / 'answer.txt')
shutil.copy2(ANSWER_ZIP, SUB_DRIVE / 'answer.zip')
print(f'mirrored → {SUB_DRIVE}')


In [ ]:
# === Format validation (match Document/answer.txt) ===
lines = ANSWER_TXT.read_text().strip().split('\n')
assert len(lines) == 1978, f'expected 1978 lines, got {len(lines)}'

gallery_set = set(gallery_ids)
bad = []
for i, line in enumerate(lines):
    toks = line.split()
    if len(toks) != 10:
        bad.append((i, f'len={len(toks)}'));                continue
    if len(set(toks)) != 10:
        bad.append((i, 'duplicate IDs in row'));            continue
    miss = [t for t in toks if t not in gallery_set]
    if miss:
        bad.append((i, f'unknown ids: {miss[:3]}'));        continue
assert not bad, f'{len(bad)} bad lines; first: {bad[:5]}'
print('✓ submission OK — 1978 rows × 10 unique gallery IDs')

# Quick comparison vs Document/answer.txt sample (if present locally)
sample = Path('/home/bao/Documents/AIC2026/Document/answer.txt')
if sample.exists():
    sample_lines = sample.read_text().strip().split('\n')
    assert len(sample_lines) == 1978
    top1_match = sum(1 for a, b in zip(sample_lines, lines)
                     if a.split()[0] == b.split()[0])
    print(f'top-1 match vs Document/answer.txt: {top1_match}/{len(lines)} '
          f'({100*top1_match/len(lines):.1f}%)')
wait_for_pending_syncs()
print('Done.')
